In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from dotenv import load_dotenv
from accelerate import Accelerator
from HuggingFaceModel import HuggingFaceModel
from TrainStrategy import TrainStrategy
from constant import *
from LlmOutputLabelConverter import LlmOutputLabelConverter
from SatdToYesNoConverter import SatdToYesNoConverter

In [ ]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Prompts

In [ ]:
definition_prompt = PromptTemplate(
    name="definition",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments.",
    instruction="Assign the label of SATD or Not-SATD for each given source code comment.\n\nHere are some examples:",
    n_shot_template='### Comment text: """ {{ text }} """',
    n_shot_answer_template="### Label: {{ label }}\n\n",
    line_m_before=0,
    line_n_after=0,
    add_question_label=True)

mat_prompt = PromptTemplate(
    name="mat",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contains specific keywords: TODO, FIXME, HACK, and XXX.",
    instruction="Assign the label of SATD or Not-SATD for each given source code comment.\n\nHere are some examples:",
    n_shot_template='### Comment text: """ {{ text }} """',
    n_shot_answer_template="### Label: {{ label }}\n\n",
    line_m_before=0,
    line_n_after=0,
    add_question_label=True)

jitterbug_prompt = PromptTemplate(
    name="jitterbug",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contains specific keywords: TODO, FIXME, HACK, and WORKAROUND.",
    instruction="Assign the label of SATD or Not-SATD for each given source code comment.\n\nHere are some examples:",
    n_shot_template='### Comment text: """ {{ text }} """',
    n_shot_answer_template="### Label: {{ label }}\n\n",
    line_m_before=0,
    line_n_after=0,
    add_question_label=True)
gpt4_prompt = PromptTemplate(
    name="gpt",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contains specific keywords: TODO, FIXME, HACK, XXX, NOTE, DEBT, REFACTOR, OPTIMIZE, TEMP, WORKAROUND, KLUDGE, REVIEW, NOFIX, PENDING, and BUG.",
    instruction="Assign the label of SATD or Not-SATD for each given source code comment.\n\nHere are some examples:",
    n_shot_template='### Comment text: """ {{ text }} """',
    n_shot_answer_template="### Label: {{ label }}\n\n",
    line_m_before=0,
    line_n_after=0,
    add_question_label=True)

In [ ]:
output_label_converter = SatdToYesNoConverter(LlmOutputLabelConverter({'SATD', 'Not-SATD'}, 'Not-SATD'))
prompt_templates = [definition_prompt, mat_prompt, jitterbug_prompt, gpt4_prompt]
shots_df = detect_n_shot_df.copy(deep=True)
shots_df['label'] = shots_df['label'].map({'yes': 'SATD', 'no': 'Not-SATD'})
shots_dataset = Dataset.from_pandas(shots_df)

In [ ]:
for prompt_template in prompt_templates:
    for model_name in ['google/flan-t5-small', 'google/flan-t5-base', 'google/flan-t5-large', 'google/flan-t5-xl',
                       'google/flan-t5-xxl']:
        for shots in [2 * n for n in range(6)]:
            t5_model = HuggingFaceModel('detect', model_name, output_label_converter, True)
            t5_model.fit(shots_dataset)
            t5_model.predict(detect_test_dataset, DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots,
                             verbose=False)


# Dry Run

In [ ]:
for model_name in ['google/flan-t5-small']:
    t5_model = HuggingFaceModel('detect', model_name, output_label_converter, True)
    t5_model.fit(shots_dataset)
    t5_model.predict(detect_test_dataset.select(range(1000)), DATASET_NAME, definition_prompt, TrainStrategy.N_SHOT_TOP, 0,
                     verbose=False)
